### Statistic for P117
#### for some programs we started in April

In [1]:
import astroquery
from astroquery.eso import Eso
import pandas as pd
import numpy as np
import sys
import math
from numpy import *
from astropy.table import Table
from astropy.table import *
from astropy.io import ascii
from datetime import datetime, date, timedelta
from termcolor import colored

eso=Eso()
eso.ROW_LIMIT = -1 


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns',None)

#### Defines a general search for available data in the ESO archive
#### - using a date range, PIDs and instrument as input¶

In [8]:
def eso_query(instr,start,end,pid):
    num_obs= 0
    tot_exptime = 0.0
    table = eso.query_main(column_filters={'instrument':instr,'dp_cat': 'SCIENCE','stime':start,'etime':end,'prog_id':pid})
    if not table:
        num_Obs      = num_obs + 0
        tot_exptime  = 0.0
    else:
        num_Obs = num_obs + len(table)
        pd_table    = table.to_pandas()
        tot_exptime = pd_table["Exptime"].sum()
    
    return num_Obs,tot_exptime


def eso_query_target(instr,target,start,end,pid):
    num_obs= 0
    tot_exptime_ir = 0.0
    tot_exptime_ir = 0.0
    table = eso.query_main(column_filters={'instrument':instr,'object':target_,'dp_cat': 'SCIENCE','stime':start,'etime':end,'prog_id':pid})
    if not table:
        num_Obs      = num_obs + 0
        tot_exptime  = 0.0
    else:
        num_Obs = num_obs + len(table)
        pd_table    = table.to_pandas()
        tot_exptime = pd_table["Exptime"].sum()
    
    return num_Obs,tot_exptime


In [30]:
info = "/home/angela/LaSilla/P117/P117_pi.list"

data = open(info,'r')

#start   = input('Start date of query (yyyy-mm-dd):  ')
#end     = input('End date of query (yyyy-mm-dd):    ')
start_1 = "2026-05-01"
end_1 = (datetime.now()+timedelta(days=2)).strftime('%Y-%m-%d')             # date for tomorrow, to make sure that all nights are  used
 
end_f   = (datetime.now() + timedelta(days=-1) ).strftime('%Y-%m-%d')     # date for yesterday, the day the last observing night started
#end_f   = end.strftime('%Y-%m-%d')

print("     ")
print("     ")
print("     ")

orig_stdout = sys.stdout

path = "/home/angela/LaSilla/P117"

file_out   = path + "/Statistic/P117_stats.cvs"
file_out_2 = path + "/Statistic/P117_stats_a.cvs"

data_2 = []
col1=[]    # full name
col2=[]    # list of PIDs
col3=[]    # instrument
col4=[]    # allocated time in hours Category A
col5=[]    #  requested time in proposal
col6=[]    # number of files
col7=[]    # exposure times
col8=[]    # execution time in hr
col9=[]    # fraction in %
col10 =[]  # fraction completed in of total %


data_3 = []
cola=[]    # full name
colb=[]    # 1st PID
colc=[]    # 2nd PID
cold=[]    # instrument
cole=[]    # allocated time A in hr
colf=[]    # tie asked for in proposal
colg=[]    # number of files
colh=[]    # exposure time in hr
coli=[]    # observed time in hr
colj=[]    # fraction completed in %
colk=[]    # fraction of total time asked for in proposal

for line in data:    
    num = 0
    tot_exptime = 0.0
    tot_time    = 0.0
    
    line     = line.strip()
    columns  = line.split()
    name     = columns[0]
    first    = columns[1]
    email    = columns[2]
    instr    = columns[3]
    pid_1    = columns[4]
    pid_2    = columns[5]
    t_tot_A  = float(columns[6])  
    t_tot_B  = float(columns[7])
    tot_alloc_time = t_tot_A  + t_tot_B 

    f_name = name+','+ first
    full_name = ("{:24}".format(f_name))
    print(full_name,t_tot_A,t_tot_B)
    idlist = pid_1

    if pid_2 != str("000.0000.000"):
        idlist = idlist+", "+pid_2

# Check FEROS programs
    if instr  == "FEROS":
        if name == "Henning":
            start_1 = "2026-04-22"
        num         = 0
        tot_exptime = 0.0
        tot_time    = 0.0
# check before dome repair        
        result_1_a = eso_query(instr,start_1,end_1,pid_1)
        num = num+result_1_a[0]
        tot_exptime= tot_exptime+result_1_a[1]
        result_1_b = eso_query(instr,start_1,end_1,pid_2)
        num = num+result_1_b[0]
        tot_exptime= tot_exptime+result_1_b[1]
  
#    Check WFI programs
    if instr == "WFI" :
        tot_time= 0.0
        if name =="BANADOS":
            oh_time      = 226.0            # new overhead, 3.5 min for fetch... + setup + readout
            num          = 0
            tot_time     = 0.0
            tot_exptime  = 0.0

            result_1_a   = eso_query(instr,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
            tot_exptime  = tot_exptime+result_1_a[1]
            result_1_b   = eso_query(instr,start_1,end_1,pid_2)
            num          = num+result_1_b[0]
#            print("Number of files bevore dome repair: ", num     )
            tot_exptime  = tot_exptime+result_1_a[1]+ result_1_a[1]

# Check after dome repair was finished
            result_2_a   = eso_query(instr,start_2,end_2,pid_1)
            num          = num+result_2_a[0]
            tot_exptime  = tot_exptime+result_2_a[1]
            result_2_b   = eso_query(instr,start_2,end_2,pid_2)
            num          = num+result_2_b[0]
            tot_exptime  = (tot_exptime+result_2_b[1])/3600.0
            tot_time     = ((num*oh_time)/3600.0)  + tot_exptime          
        
        
        if name == "SUYU":
            num          = 0
            tot_exptime  = 0.0
            tot_time     = 0.0
            oh_time      = 610.0                                       # per 4 images
            result_1_a   = eso_query(instr,start_1,end_1,pid_1)
            num          = num+result_1_a[0]
            tot_exptime  = tot_exptime+result_1_a[1]

# Check after dome repair was finished
            result_1_b   = eso_query(instr,start_2,end_2,pid_1)
            num          = num+result_1_b[0]
            tot_exptime  = tot_exptime+result_1_b[1]

            #Check the second PID 
            result_2_a   = eso_query(instr,start_1,end_1,pid_2)
            num          = num+result_2_a[0]
            tot_exptime  = tot_exptime+result_2_a[1]
            result_2_b   = eso_query(instr,start_2,end_2,pid_2)
            num          = num+result_2_b[0]
            tot_exptime  = (tot_exptime+result_2_b[1] )/3600.0
            
            tot_time     = ((num/4*oh_time)/3600.0)  + tot_exptime


###################################################################################

#### Calculate percentages of allocated times

    time_perc_A = 0.0                                                       # percentage of Cat A. time observed
    time_perc_B = 0.0                                                       # percentage of Cat B. time observed
    
# Case 1: only category A is assigned, and total execution time is greater than 0.0:
    
    if tot_time > 0.0 and t_tot_A > 0.0 and t_tot_B == 0.0:
        time_perc_A = (100.0* (tot_time /t_tot_A))
        time_perc_B = 0.0
    elif tot_time > 0.0 and t_tot_A > 0.0 and t_tot_B > 0.0 and tot_time < t_tot_A:   # time observed is less than Cat. A
        time_perc_A = (100.0* (tot_time /t_tot_A))
        

# Case 2: only category B is assigned, and total execution time is greater than 0.0:        
    elif  tot_time > 0.0 and t_tot_B > 0.0 and t_tot_A == 0.0:  
        time_perc_B = 100.0* (tot_time /t_tot_B)
        time_perc_A = 0.0

# Case 3: both category A & B are assigned, and the total execution time is greater than the time assigned to Category A:    
#    elif tot_time > t_tot_A and t_tot_B > 0.0  and t_tot_A !=0: 
#        time_perc_A = 100.0
#        time_perc_B = (100.0* (tot_time /t_tot_A))-100.0

# Case 4: no time has been assigned to either category (not useful):    
#    elif t_tot_A == 0 and t_tot_B == 0:
#        time_perc_A = 0.0
#        time_perc_B = 0.0

# Case 5: no observation has been done:
#    elif tot_exptime  == 0.0:
#        time_perc_A   = 0.0
#        time_perc_B   = 0.0




### Format the values
    
    t_tot_A_f      = ("{:>8.0f}".format(t_tot_A))                            # formatted allocated time cat- A
    t_tot_B_f      = ("{:>8.0f}".format(t_tot_B))                            # formatted allocated time cat- B
    tot_time_f     = ("{:>8.2f}".format(tot_time))                           # formatted observed time
    time_perc_A_f  = ("{:>6.2f}".format(time_perc_A))
    time_perc_B_f  = ("{:>6.2f}".format(time_perc_B))
    tot_exptime_f  = ("{:>6.2f}".format(tot_exptime))
#    print(name,num, tot_time_f,tot_exptime_f)

    data_2.append({'#PI_name                ': full_name.ljust(20," "),
                   'PIDs              ': idlist.ljust(30," "),
                   'Inst.': str(instr).center(5," "),
                   'Cat-A [h]': str(t_tot_A_f).rjust(10," "),
                   'Cat-B [h]': str(t_tot_B_f).rjust(10," "),
                   '# files': str(num).rjust(10," "),
                   'exp. [h]': tot_exptime_f,
                   'exec. [h] ': str(tot_time_f).center(10," "),
                   '  A [%]': time_perc_A_f,
                   '  B [%]': time_perc_B_f
        }              
    )
    col1.append(full_name)
    col2.append(idlist.ljust(30," "))
    col3.append(instr.center(9," "))
    col4.append(t_tot_A_f)
    col5.append(t_tot_B_f)
    col6.append(num)
    col7.append(tot_exptime_f)
    col8.append(tot_time_f)
    col9.append(time_perc_A_f)
    col10.append(time_perc_B_f)
              
    data_3.append({'#PI_name': full_name.ljust(20," "),
                   '       PID_1       ': pid_1.center(15," "),
                   '       PID_2       ': pid_2.center(15," "),
                   'Instrument  ': str(instr).center(5," "),
                   'Cat-A [h]': str(t_tot_A_f).rjust(10," "),
                   'Cat-B [h]': str(t_tot_B_f).rjust(10," "),
                   '# of files': str(num).rjust(10," "),
                   'exp. [h]': tot_exptime_f,
                   'exec.[h] ': str(tot_time_f).center(10," "),
                   '  A [%]': time_perc_A_f,
                   '  B.[%]': time_perc_B_f
        }              
    )
    cola.append(full_name)
    colb.append(pid_1.center(15," "))
    colc.append(pid_2.center(15," "))
    cold.append(instr.center(15," "))
    cole.append(t_tot_A_f)
    colf.append(t_tot_B_f)
    colg.append(num)
    colh.append(tot_exptime_f)
    coli.append(tot_time_f)
    colj.append(time_perc_A_f)
    colk.append(time_perc_B_f)


pd.set_option('display.precision', 1)    
df = pd.DataFrame(data_2)
df_1 = pd.DataFrame(data_3)
number = len(df.index)
#print(number)
print(df.to_string(index=False))
#print(new)
li = [df.columns.values.tolist()] + df.values.tolist()
df.to_csv(file_out, quoting=None,index=False) 
df_1.to_csv(file_out_2, index=False) 
print("    ")
print("    ")

print("Finished")
sys.stdout=orig_stdout      




ValueError: invalid literal for int() with base 10: 'A'

In [27]:
type(t_tot_A)


str